In [ ]:
%load_ext autoreload
%autoreload 2
# automatically pick up changes made in src file

# LEDGAR: multiclass softmax logistic regression
Loads the features saved by `preprocess.ipynb` (run that once first), then trains, evaluates and runs error analysis.

**Flow**
1. Imports
2. Load saved features
3. Model
4. Error analysis (validation)
5. Test (once, at the end)

## 1. Imports
All src files are imported here, once.

In [ ]:
from pathlib import Path
import sys; sys.path.append('..')     # lets the notebook find the src folder

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from src.clean import clean_text # cleaning function
from src.lemmetizer import Lemmatizer # spaCy lemmatiser class
from src.tf_idf import tfid # builds the TF-IDF vectoriser
from src.multi_class_lr_softmax import MultiClassLogisticRegression # model class
from src.classification_error_analysis import ErrorAnalysis # error analysis class

## Load the saved features (made by `preprocess.ipynb`)
No cleaning, lemmatising or TF-IDF here - everything is read from disk in seconds.
- `x_train`, `x_val`, `x_test`: TF-IDF matrices (sparse)
- `vectorizer` / `feature_names`: the fitted TF-IDF and its column (word) names
- `proc_train`, `proc_val`, `proc_test`: the lemmatised clauses with labels

In [ ]:
from scipy import sparse
import joblib

processed = Path("../data/processed")
features = Path("../data/features")

# TF-IDF matrices
x_train = sparse.load_npz(features / "x_train.npz")
x_val = sparse.load_npz(features / "x_val.npz")
x_test = sparse.load_npz(features / "x_test.npz")

# fitted vectoriser and its column (word) names
vectorizer = joblib.load(features / "vectorizer.joblib")
feature_names = vectorizer.get_feature_names_out()

# lemmatised clauses (text + labels)
proc_train = pd.read_parquet(processed / "proc_train.parquet")
proc_val = pd.read_parquet(processed / "proc_validation.parquet")
proc_test = pd.read_parquet(processed / "proc_test.parquet")

print("x_train:", x_train.shape, "| x_val:", x_val.shape, "| x_test:", x_test.shape)

## 6. Model: multiclass softmax logistic regression
- y = the label column of each split: y_train, y_val, y_test
- `label_names`: lookup from label number to clause name, e.g. 47 → "Governing Laws"

In [ ]:
y_train = proc_train.label
y_val = proc_val.label
y_test = proc_test.label

# pairs each number with its name, then makes a lookup dictionary
# zip lines up the two columns side by side
label_names = dict(zip(proc_train["label"], proc_train["label_name"]))

Call the model from the src file.
- `train()`: learn the weights
- `evaluate_train()`: score on train - compare with validation to check for overfitting
- `validate()`: score on validation - unseen data, used to choose settings
- `eval_test()`: final exam - run ONCE at the very end

In [ ]:
lr = MultiClassLogisticRegression(x_train, x_val, x_test,
                                  y_train, y_val, y_test,
                                  C= 30.0, class_weight='balanced')

lr.train() # learn weights
train_preds, train_scores = lr.evaluate_train() # the model guesses train labels, and they're marked
val_preds, val_scores = lr.validate()  

## 7. Error analysis (validation set)
Uses `src/error_analysis.py` → `ErrorAnalysis` class. Validation shows authentic weaknesses on unseen data.
1. Per-class scores, worst F1 first: which clause types are weak?
2. TP / FP / FN / TN per clause type
3. Top confused pairs: which types get mixed up with which?
4. One weak class in detail: where did its clauses go?
5. Read the actual wrong clauses
6. Confusion matrix picture (overview)
7. Group the mistakes into 3-4 reasons and count them (e.g. overlapping labels / too rare / long or merged clauses)

In [ ]:
ea = ErrorAnalysis(y_true=y_val, y_pred=val_preds,
                   label_names=label_names,
                   classes=lr.model.classes_,       # fixed order: same 100 classes in every table
                   texts=proc_val["text"])           # original clause text, to read mistakes

In [ ]:
# 1. per-class scores - worst F1 first (check the support column: are the worst ones small classes?)
ea.report_df().head(15)

In [ ]:
# 2. TP / FP / FN / TN per clause type - sort by FN = types the model misses most
ea.counts().sort_values("FN", ascending=False).head(15)

In [ ]:
# 3. top confused pairs: one row per (true, predicted) pair, most common first
ea.top_confusions(n=15)

In [ ]:
# 4. one weak class in detail - change the name to any class from step 1
ea.class_confusions("Assigns", n=5)

In [ ]:
# 5. read the actual mistakes
mistakes = ea.mistakes()
pct_mistakes = ((len(mistakes))/(len(y_val))*100)
print(len(mistakes), "wrong out of", len(y_val), "pct mistakes", pct_mistakes)
mistakes.head(20)

In [ ]:
# 6. confusion matrix picture (quick overview only - use step 3 for the details)
ea.plot_confusion_matrix(size=30)

CHANGED FROM (1,2) - NGRAMS (uni and bi) to (1,3) - NGRAMS (uni, bi, and tri)
    - this decreased the error between waivers and no waivers classes
    - no change in applicable law and governing law - this shows that there is a different discrepency between them that the model misses
    

## Error Analysis Part 2 ## 
- applicable law and governing law analysis

In [ ]:
train = pd.read_parquet("../data/interim/train_eda.parquet")
print("loaded train:", train.shape)   # expect (59933, 5)
train
overlapping = train[train['label_name'].isin(['Applicable Laws', 'Governing Laws'])]
# they are not labelled differently with the same text == no duplicates 
overlapping["text"].duplicated().any()


Check modelling coefficients - looking at the weights;

In [ ]:
# name -> label number (flip your label_names into a lookup)
# flip the label names fictionary around - the key name and then the numerical value encoded
name_to_label = {v: k for k, v in label_names.items()}
classes = list(lr.model.classes_) # 0-99

# feature_names - column headers
# classes - rows 
# values. inside the cells = coefficient 
def class_weights(name):
    row = classes.index(name_to_label[name]) # which row of coef_ co-ordinates the list of model.classes intot he labelled names
    return pd.Series(lr.model.coef_[row], index=feature_names)

gl = class_weights("Governing Laws")
al = class_weights("Applicable Laws")

# 1. top 20 words for each class, side by side
pd.DataFrame({
    "Governing Laws": gl.sort_values(ascending=False).head(20).index,
    "Applicable Laws": al.sort_values(ascending=False).head(20).index,
})

# 2. what separates them: positive = pushes to GL, negative = pushes to AL
diff = (gl - al).sort_values()
print(diff.tail(20))   # most "Governing Laws" words
print(diff.head(20))   # most "Applicable Laws" words

The more positive values: show that the result is governing law, more negative: applicable law:
- count what % of applicable law contains govern; what % of governing contains applicable law, if both are high , an overlap is evident 


In [ ]:
gov_df = proc_train[proc_train['label'].isin([47])]
app_df = proc_train[proc_train['label'].isin([4])]
app_df[app_df['clean_text'].str.contains(r'\bgovern(?:s|ed|ing)?\b')] # 258
gov_df[gov_df['clean_text'].str.contains('applicable law', case = False)] # 113
len(app_df) # 349
len(gov_df)  #3157


About three-quarters of Applicable Laws clauses are choice-of-law clauses: the same kind of clause as Governing Laws, filed under a different heading.

74% of Applicable Laws clauses are choice-of-law clauses with the same wording as Governing Laws. The labels come from drafters’ headings, so the two classes overlap.

## Test

In [ ]:
test_preds, test_scores = lr.eval_test()  


In [ ]:
ea_test = ErrorAnalysis(y_true=y_test, y_pred=test_preds,
                   label_names=label_names,
                   classes=lr.model.classes_,       # fixed order: same 100 classes in every table
                   texts=proc_test["text"])           # original clause text, to read mistakes

In [ ]:
display([ea_test.counts().sort_values("FN", ascending=False).head(15),
         ea_test.top_confusions(n=15),
         ea_test.class_confusions("Assigns", n=5)])


In [ ]:
mistakes_test = ea_test.mistakes()
pct_mistakes_test = ((len(mistakes_test))/(len(y_test))*100)
print(len(mistakes_test), "wrong out of", len(y_test), "pct mistakes", pct_mistakes_test)

mistakes_test.head(20)
ea_test.plot_confusion_matrix(size=30)